# Vaccines: your own protections

Immune's built-in threats cover what every LLM app shares. **Vaccines** cover what only yours has: a competitor you
must never recommend, an account-number format customers shouldn't paste, a refund limit, advice your assistant
shouldn't give. A vaccine is a small YAML file; it compiles into the same pipeline, verdicts and dashboards as the
built-in threats.

| I want to catch… | Detector | Section |
| --- | --- | --- |
| Specific words or names | `keywords` | [1](#1.-Keywords) |
| The same, but only when it really means what the rule is about | `keywords` + `confirm: jev` | [2](#2.-Let-Jev-confirm-each-match) |
| A text format, such as account numbers | `regex` | [3](#3.-Regular-expressions) |
| Something only meaning can tell | `questions` for Jev | [4](#4.-Questions-for-Jev) |
| A tool call with certain arguments | `tool` + `argument` | [5](#5.-Tool-rules) |
| Anything you can write in Python | `python` | [6](#6.-Python-functions) |
| A problem many apps share, already measured | the vaccine library | [9](#9.-Vaccines-that-ship-with-Immune) |

Everything here ships in the package: your vaccines live in your own repository, and you don't need to contribute
anything to Immune. Every vaccine starts **observed**, like built-in detectors. Add `enforcement: enforce` (or list it
in a site's `enforce`) to act on it. The [vaccines guide](../../docs/guides/vaccines.md#build-a-vaccine-for-your-app)
walks through the whole workflow and documents every field. To design one for your app step by step, see
[Designing and testing your own vaccines](12b_designing_vaccines.ipynb).

## 1. Keywords

**Before:** a support assistant that sometimes recommends a competitor.

In [1]:
from openai import OpenAI

client = OpenAI()
SYSTEM = (
    "You are the support assistant for Pinecrest Outfitters. We don't sell kayaks or canoes; "
    "TrailMart down the road has a good paddling section."
)
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Do you sell kayaks?"}],
)
print(response.choices[0].message.content)

We don’t sell kayaks or canoes at Pinecrest Outfitters. TrailMart down the road has a good paddling section.


**After:** a keyword vaccine, enforced, with the message customers should see instead. In your project the YAML lives
in a `vaccines/` folder and `immune.yaml` lists it (`vaccines: {paths: [vaccines/]}`); here it is written to a
temporary folder and passed to `immune.init(vaccines=[...])`.

In [2]:
import tempfile
from pathlib import Path

from openai import OpenAI

import immune

vaccines = Path(tempfile.mkdtemp())
(vaccines / "no_competitors.yaml").write_text("""
id: pinecrest.no_competitor_mentions
title: The reply names a competitor
stage: output
detect:
  keywords: ["TrailMart", "Summit Supply"]
respond:
  message: "I can only help with Pinecrest Outfitters products. Is there anything else I can find for you?"
enforcement: enforce
""")
immune.init(vaccines=[vaccines])
client = OpenAI()
SYSTEM = (
    "You are the support assistant for Pinecrest Outfitters. We don't sell kayaks or canoes; "
    "TrailMart down the road has a good paddling section."
)
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Do you sell kayaks?"}],
)
print(response.choices[0].message.content)
verdict = immune.verdict(response)
print("\nverdict:", verdict.action, "| evidence:", [hit.evidence for hit in verdict.hits])

immune: sensor unavailable for input (sensor exceeded 0.80s); floor candidates act without Jev


I can only help with Pinecrest Outfitters products. Is there anything else I can find for you?

verdict: rewrite | evidence: [("keyword 'TrailMart'",)]


## 2. Let Jev confirm each match

Keywords are blunt: "we're across the road from TrailMart" names a competitor too, and is perfectly fine. Add
`confirm: jev` and each match becomes a *candidate*: Jev is asked whether it is a real case of the rule described by
the vaccine's `title` and `description`, and the vaccine fires only if Jev agrees.

In [3]:
import tempfile
from pathlib import Path

from openai import OpenAI

import immune

vaccines = Path(tempfile.mkdtemp())
(vaccines / "no_competitors.yaml").write_text("""
id: pinecrest.no_competitor_recommendations
title: The reply recommends a competitor
description: The reply suggests the customer shop at a rival store instead of Pinecrest Outfitters.
stage: output
detect:
  keywords: ["TrailMart", "Summit Supply"]
  confirm: jev
respond:
  message: "I can only help with Pinecrest Outfitters products. Is there anything else I can find for you?"
enforcement: enforce
""")
immune.init(vaccines=[vaccines])
client = OpenAI()
SYSTEM = (
    "You are the support assistant for Pinecrest Outfitters, at 12 Pine Street, across the road from TrailMart. "
    "We don't sell kayaks or canoes; TrailMart has a good paddling section."
)
for question in ["Do you sell kayaks?", "How do I find your store? Is it near TrailMart?"]:
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
    )
    verdict = immune.verdict(response)
    print(f"customer: {question}\nreply:    {response.choices[0].message.content[:160]}")
    print(f"verdict:  {verdict.action}  {[list(hit.evidence) for hit in verdict.hits]}\n")

customer: Do you sell kayaks?
reply:    I can only help with Pinecrest Outfitters products. Is there anything else I can find for you?
verdict:  rewrite  [["keyword 'TrailMart'", 'pinecrest_no_competitor_recommendations__confirmed=0.84']]



customer: How do I find your store? Is it near TrailMart?
reply:    Yes — we’re at **12 Pine Street**, and we’re **across the road from TrailMart**. If you’re heading to TrailMart, just look across the street for Pinecrest Outfi
verdict:  allow  []



**What you see:** the recommendation is replaced; the directions pass. The evidence shows the keyword match and
Jev's answer (`...__confirmed=0.xx`).

## 3. Regular expressions

Customers paste things they shouldn't. An input vaccine catches them before the model sees them. Patterns must use
bounded repetition (`{8}`, `{1,40}`), so no vaccine can hang on long text.

In [4]:
import tempfile
from pathlib import Path

from openai import OpenAI

import immune

vaccines = Path(tempfile.mkdtemp())
(vaccines / "no_account_numbers.yaml").write_text("""
id: pinecrest.no_account_numbers
title: A loyalty account number pasted into the chat
stage: input
detect:
  regex: ['\\bPC-\\d{8}\\b']
respond:
  text: redirect
  message: "For your security, please don't share account numbers here. Sign in to see your points."
enforcement: enforce
""")
immune.init(vaccines=[vaccines])
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters."},
        {"role": "user", "content": "My loyalty account is PC-48213377. How many points do I have?"},
    ],
)
print(response.choices[0].message.content)
print("verdict:", immune.verdict(response).action)

For your security, please don't share account numbers here. Sign in to see your points.
verdict: redirect


## 4. Questions for Jev

Some rules can only be judged by meaning. A store's shopping assistant shouldn't give first-aid advice, however
helpful it tries to be. Write the rule as a yes/no statement for Jev; a small head turns the answer into a probability,
and `threshold` sets when the vaccine fires.

In [5]:
import tempfile
from pathlib import Path

from openai import OpenAI

import immune

vaccines = Path(tempfile.mkdtemp())
(vaccines / "no_medical_advice.yaml").write_text("""
id: pinecrest.no_medical_advice
title: The shopping assistant gives medical advice
stage: output
detect:
  questions:
    - key: medical_advice
      text: "The assistant output tells the user how to treat an injury, illness or symptom."
  threshold: 0.8
respond:
  message: "I can't give medical advice; please talk to a medical professional. First-aid kits are under Safety."
enforcement: enforce
""")
immune.init(vaccines=[vaccines])
client = OpenAI()
SYSTEM = "You are the shopping assistant for Pinecrest Outfitters, a camping and hiking store."
for question in ["I got a bad blister on yesterday's hike. How should I treat it?", "What's your returns window?"]:
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
    )
    verdict = immune.verdict(response)
    print(f"customer: {question}\nreply:    {response.choices[0].message.content[:200]}\nverdict:  {verdict.action}\n")

customer: I got a bad blister on yesterday's hike. How should I treat it?
reply:    I can't give medical advice; please talk to a medical professional. First-aid kits are under Safety.
verdict:  rewrite



customer: What's your returns window?
reply:    Most items can be returned within **30 days of purchase** with a receipt or order confirmation, as long as they’re unused and in original condition.

A few exceptions may apply for:
- **Clearance or f
verdict:  allow



## 5. Tool rules

Tool rules match a tool name (globs allowed) and optionally its arguments. `respond: {tool: confirm}` asks the user
first on user-facing sites. This example uses Immune's test harness with a scripted model, so the tool calls are the
same on every run:

In [6]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, FakeToolCall, ImmuneHarness

vaccines = Path(tempfile.mkdtemp())
(vaccines / "refund_limit.yaml").write_text("""
id: pinecrest.refund_over_limit
title: Refunds over $100 need the customer's confirmation
stage: tool
detect:
  tool: issue_refund
  argument: {path: amount, greater_than: 100}
respond:
  tool: confirm
enforcement: enforce
""")
for amount in [40, 250]:
    harness = ImmuneHarness(
        Path(tempfile.mkdtemp()),
        script=FakeReply(tool_calls=[FakeToolCall("issue_refund", {"order_id": "5521", "amount": amount})]),
        config={"vaccines": {"paths": [str(vaccines)]}, "sites": {"support-agent": {"user_facing": True}}},
    )
    with immune.site("support-agent"):
        response = harness.openai().chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role": "user", "content": f"Please refund ${amount} for order 5521."}],
            tools=[{"type": "function", "function": {"name": "issue_refund", "description": "Refund an order."}}],
        )
    message = response.choices[0].message
    print(f"refund ${amount}: verdict={harness.verdict(response).action}")
    print(f"    tool calls kept: {[call.function.name for call in message.tool_calls or []]}")
    print(f"    text: {message.content}")
    harness.close()

refund $40: verdict=allow
    tool calls kept: ['issue_refund']
    text: None
refund $250: verdict=confirm
    tool calls kept: []
    text: Before I do that, please confirm: issue_refund(amount=250, order_id='5521'). Reply "yes" to continue. (ref 47d28ffe32)


## 6. Python functions

When a rule needs your own data or logic, point a vaccine at a function: `python: module.path:function_name`. It
receives a `VaccineContext` and returns evidence (or `True`) when the threat is present. It runs in the request path,
so keep it fast and free of side effects; `budget_ms` logs slow calls.

In [7]:
import sys
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, FakeToolCall, ImmuneHarness

project = Path(tempfile.mkdtemp())
(project / "pinecrest_rules.py").write_text("""
from immune.vaccines import VaccineContext

VIP_ACCOUNTS = {"vip-1001", "vip-2002"}


def touches_vip_account(context: VaccineContext) -> str | None:
    account = str(context.arguments.get("account", ""))
    return f"account {account} is a VIP account" if account in VIP_ACCOUNTS else None
""")
sys.path.insert(0, str(project))  # in your app, the module is simply importable
(project / "vip.yaml").write_text("""
id: pinecrest.vip_accounts
title: Changes to VIP accounts go to a person
stage: tool
detect:
  python: pinecrest_rules:touches_vip_account
  budget_ms: 20
respond:
  tool: hold
  message: VIP accounts are handled by the account team.
enforcement: enforce
""")
for account in ["cust-3003", "vip-1001"]:
    harness = ImmuneHarness(
        Path(tempfile.mkdtemp()),
        script=FakeReply(tool_calls=[FakeToolCall("update_account", {"account": account, "points": 500})]),
        config={"vaccines": {"paths": [str(project / "vip.yaml")]}},
    )
    response = harness.openai().chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": f"Add 500 points to {account}."}],
        tools=[{"type": "function", "function": {"name": "update_account", "description": "Update an account."}}],
    )
    verdict = harness.verdict(response)
    print(f"{account}: verdict={verdict.action} evidence={[list(hit.evidence) for hit in verdict.hits]}")
    harness.close()

cust-3003: verdict=allow evidence=[]
vip-1001: verdict=hold evidence=[['account vip-1001 is a VIP account']]


## 7. Test, trial and generate vaccines

Put examples in the file: `tests.positives` must fire and `tests.negatives` must not. Then:

| Command | What it does |
| --- | --- |
| `immune vaccines test [paths]` | Runs each file's examples through a private runtime |
| `immune vaccines trial <id or file>` | Estimates how often it fires on everyday traffic (packaged samples, plus `--corpus`) |
| `immune vaccinate <id> --positive … --negative …` | Builds a vaccine from examples, tests and trials it, and writes it observed-first |
| `immune vaccines list` | Every protection, built-in and yours, and whether it is on |

(In a terminal, drop the `!`.)

In [8]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())
(folder / "no_competitors.yaml").write_text("""
id: pinecrest.no_competitor_mentions
title: The reply names a competitor
stage: output
detect:
  keywords: ["TrailMart", "Summit Supply"]
tests:
  positives: ["You could try TrailMart for kayaks."]
  negatives: ["We stock two-person tents and sleeping pads."]
""")

!immune vaccines test {folder}
!immune vaccines trial {folder}/no_competitors.yaml --show 3

vaccine                           expect  example                                       result  detail
────────────────────────────────  ──────  ────────────────────────────────────────────  ──────  ────────────────────────────────
pinecrest.no_competitor_mentions  fire    You could try TrailMart for kayaks.           pass    fired p=1.00 keyword 'TrailMart'
pinecrest.no_competitor_mentions  quiet   We stock two-person tents and sleeping pads.  pass    did not fire

1 vaccines, 2 examples: all passed


trial pinecrest.no_competitor_mentions (offline): 40 everyday output samples
fired on 0 of 40 samples (0.0%)


In [9]:
import tempfile
from pathlib import Path

folder = Path(tempfile.mkdtemp())

!immune vaccinate pinecrest.no_lifetime_warranty --stage output --keyword "lifetime warranty" \
    --positive "All our tents come with a lifetime warranty." \
    --negative "Tents have a 2-year warranty against manufacturing defects." \
    --message "Our tents have a 2-year warranty against manufacturing defects." --dir {folder}
!cat {folder}/pinecrest.no_lifetime_warranty.yaml

vaccinate pinecrest.no_lifetime_warranty
────────────────────────────────────────
vaccine                         expect  example                                                      result  detail
──────────────────────────────  ──────  ───────────────────────────────────────────────────────────  ──────  ────────────────────────────────────────
pinecrest.no_lifetime_warranty  fire    All our tents come with a lifetime warranty.                 pass    fired p=1.00 keyword 'lifetime warranty'
pinecrest.no_lifetime_warranty  quiet   Tents have a 2-year warranty against manufacturing defects.  pass    did not fire



trial pinecrest.no_lifetime_warranty (offline): 40 everyday output samples
fired on 0 of 40 samples (0.0%)

wrote /tmp/tmp5837ffai/pinecrest.no_lifetime_warranty.yaml (enforcement: observe)
next: add /tmp/tmp5837ffai/ to vaccines.paths, watch its observed hits, then set enforcement: enforce


# Created by `immune vaccinate` on 2026-09-30. It starts observed.
id: pinecrest.no_lifetime_warranty
version: 1.0.0
title: No lifetime warranty
stage: output
detect:
  keywords:
  - lifetime warranty
respond:
  message: Our tents have a 2-year warranty against manufacturing defects.
enforcement: observe
tests:
  positives:
  - All our tents come with a lifetime warranty.
  negatives:
  - Tents have a 2-year warranty against manufacturing defects.
provenance:
  created_by: immune vaccinate


## 8. Switch built-in protections off or on

The same switches work for built-in threats and vaccines, globally or per site, and live with `immune.configure()`.
Switching off a floor rule (F1–F10) is refused unless you say so explicitly:

In [10]:
import immune

immune.init(config={"vaccines": {"disabled": ["output.task_deviation"]}})
print("output.task_deviation switched off globally")

immune.init(config={"sites": {"playground": {"vaccines": {"disabled": ["input.off_task"]}}}})
print("input.off_task switched off at the playground site")

try:
    immune.init(config={"vaccines": {"disabled": ["output.secret_leak"]}})
except immune.ConfigError as error:
    print("\nConfigError:", error)

output.task_deviation switched off globally
input.off_task switched off at the playground site

ConfigError: these settings would disable floor protections: vaccines.disabled: output.secret_leak (F3). Set vaccines.allow_floor_changes: true if this is intended


## 9. Vaccines that ship with Immune

Some problems are shared by many apps: a shopping assistant shouldn't tell people how much medicine to take, and a
bank's support bot shouldn't recommend investments. Immune ships a library of vaccines for these, written and
measured in the project's laboratory. Every one is **off** until you switch it on, and **observed** until you
enforce it:

```yaml
vaccines:
  enabled: [immune.finance.personal_investment_advice]    # everywhere
sites:
  pharmacy-chat:
    vaccines: {enabled: [immune.health.*]}                 # a whole domain, at one site
    enforce: [immune.health.dosage_instructions]
```

```bash
immune vaccines list --library                             # what's available, and whether it's on
immune vaccines show immune.health.dosage_instructions     # its measured card
immune vaccines fork immune.health.dosage_instructions --as acme.dosage_instructions   # tailor a copy
```

`immune.configure(vaccines={"enabled": [...]})` switches them on live. A vaccine that's off costs nothing: no Jev
questions and no detector time. The [vaccine catalog](../../docs/reference/vaccine-catalog.md) lists them with their
measured numbers.

## Recap

- A vaccine is a YAML file: an id, a stage, one detector and a response
- `keywords`, `regex`, `questions`, `tool` rules and `python` functions cover every kind of rule
- `confirm: jev` lets Jev confirm each keyword, regex, tool or Python match
- `immune vaccines test`, `trial` and `vaccinate` keep them honest in CI
- Vaccines start observed; `enforcement: enforce` or a site's `enforce` list turns them on
- The vaccine library ships measured vaccines for problems many apps share: switch them on with `vaccines.enabled`

Next: [Designing and testing your own vaccines](12b_designing_vaccines.ipynb)